# ಪಾಠ 11 - ಏಜೆಂಟ್-ಟು-ಏಜೆಂಟ್ (A2A) ಪ್ರೋಟೋಕಾಲ್


## ಸೆಟಪ್


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A ಪ್ರೋಟೋಕಾಲ್ ಎಂದರೆ ಏನು?

The **ಏಜೆಂಟ್-ಟು-ಏಜೆಂಟ್ (A2A) ಪ್ರೋಟೋಕಾಲ್** ಒಂದು open standard ಆಗಿದ್ದು ಎಐ ಏಜೆಂಟ್‌ಗಳಿಗೆ ಪರಸ್ಪರ ಸಂವಹನ ನಡೆಸಲು, ಪರಸ್ಪರವನ್ನು ಕಂಡುಹಿಡಿಯಲು ಮತ್ತು ಸಹಕರಿಸಲು ಅವಕಾಶ ನೀಡುತ್ತದೆ — ಇವು ವಿಭಿನ್ನ ಫ್ರೇಮ್ವರ್ಕ್‌ಗಳಲ್ಲಿ ನಿರ್ಮಿಸಲ್ಪಟ್ಟಿದ್ದರೂ ಅಥವಾ ವಿಭಿನ್ನ ಸೇವೆಗಳಲ್ಲಿ ಹೋಸ್ಟ್ ಮಾಡಲ್ಪಟ್ಟಿದ್ದರೂ ಸಹ.

Key concepts:

- **ಶೋಧನೆ** – Agents publish an *ಏಜೆಂಟ್ ಕಾರ್ಡ್* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **ಸಂದೇಶ ವಿನಿಮಯ** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **ಕಾರ್ಯ ಜೀವನಚಕ್ರ** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

ಈ ಪಾಠದಲ್ಲಿ ನಾವು ಮೂರು ತಜ್ಞ ಪ್ರಯಾಣ ಏಜೆಂಟ್‌ಗಳನ್ನು ಒಂದು ಕಾರ್ಯಪ್ರವಾಹದಲ್ಲಿ ಜೋಡಿಸಿ A2A-ಶೈಲಿಯ ಸಹಕಾರವನ್ನು ಅನುಕರಿಸುತ್ತೇವೆ, ಅಲ್ಲಿ ಪ್ರತಿಯೊಬ್ಬ ಏಜೆಂಟ್ ತನ್ನ ಪರಿಣತಿಯನ್ನು ಕೊಡುಗೆ ನೀಡುತ್ತಾನೆ ಮತ್ತು ಫಲಿತಾಂಶವನ್ನು ಮುಂದಿನದಕ್ಕೆ ಹಸ್ತಾಂತರಿಸುತ್ತಾನೆ.


## ವಿಶೇಷೀಕೃತ ಪ್ರಯಾಣ ಏಜೆಂಟ್‌ಗಳನ್ನು ರಚಿಸುವುದು


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ವರ್ಕ್‌ಫ್ಲೋ ಮೂಲಕ ಬಹು-ಏಜೆಂಟ್ ಸಹಕಾರ

ನಾವು ಮೂರು ಏಜೆಂಟ್‌ಗಳನ್ನು A2A ಸಂದೇಶ ವಿನಿಮಯವನ್ನು ಪ್ರತಿಬಿಂಬಿಸುವ ಕ್ರಮಾನುಕ್ರಮದ ವರ್ಕ್‌ಫ್ಲೋವೊಂದರಲ್ಲಿ ಸಂಪರ್ಕಿಸುತ್ತೇವೆ:

1. **CurrencyExchangeAgent** ಬಳಕೆದಾರರ ವಿನಂತಿಯನ್ನು ಸ್ವೀಕರಿಸಿ ಕರೆನ್ಸಿ ಮಾರ್ಗದರ್ಶನವನ್ನು ನೀಡುತ್ತದೆ.
2. **ActivityPlannerAgent** ಸಮೃದ್ಧಿಸಿದ ಸನ್ನಿವೇಶವನ್ನು ಸ್ವೀಕರಿಸಿ ಚಟುವಟಿಕೆ ಶಿಫಾರಸುಗಳನ್ನು ಸೇರಿಸುತ್ತದೆ.
3. **TravelManagerAgent** ಎರಡೂ ಇನ್‌ಪುಟ್‌ಗಳನ್ನು ಸಂಶ್ಲೇಷಿಸಿ ಅಂತಿಮ ಪ್ರಯಾಣ ಸಂಕ್ಷಿಪ್ತವನ್ನು ರಚಿಸುತ್ತದೆ.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ಉತ್ಪಾದನಾ ಪರಿಸರದಲ್ಲಿ A2A ಅನ್ನು ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವುದು

ಉತ್ಪಾದನಾ ಪರಿಸರದಲ್ಲಿ A2A ಪ್ರೋಟೋಕಾಲ್ ಸೇವೆಗಳ ನಡುವೆ ಶಕ್ತಿಶಾಲಿ ಸಂದರ್ಭಗಳನ್ನು ಸಾಧ್ಯಮಾಡುತ್ತದೆ:

| Capability | Description |
|---|---|
| **ಫ್ರೇಮ್‌ವರ್ಕ್‌ಗಳ ನಡುವೆ ಪರಸ್ಪರ ಕಾರ್ಯನಿರ್ವಹಣೆ** | ಒಂದು ಫ್ರೇಮ್‌ವರ್ಕ್‌ನಲ್ಲಿ ನಿರ್ಮಿಸಲಾದ ಏಜೆಂಟ್ ಇನ್ನೊಂದು A2A-ಅನುಸಾರ ಫ್ರೇಮ್‌ವರ್ಕ್‌ನಲ್ಲಿ ನಿರ್ಮಿತ ಯಾವುದಾದರೂ ಏಜೆಂಟ್‌ಗೆ ಕಾರ್ಯಗಳನ್ನು ನಿಯೋಜಿಸಬಹುದು, ಇದರಿಂದ ವಿವಿಧ ಸಂಸ್ಥೆಗಳ ನಡುವಿನ ನಿಜವಾದ ಪರಸ್ಪರ ಕಾರ್ಯನಿರ್ವಹಣೆ ಸಾಧ್ಯವಾಗುತ್ತದೆ. |
| **ಸೇವಾ ಗಡಿಗಳು** | ಏಜೆಂಟ್‌ಗಳು ವಿಭಿನ್ನ ಮೈಕ್ರೋಸರ್ವೀಸ್‌ಗಳಲ್ಲಿ, ಕ್ಲೌಡ್ ಪ್ರದೇಶಗಳಲ್ಲಿ ಅಥವಾ ವಿಭಿನ್ನ ಸಂಸ್ಥೆಗಳಲ್ಲಿ ಇರಬಹುದು, ಆದರೆ ಇನ್ನೂ ಸುಗಮವಾಗಿ ಸಹಕರಿಸಬಹುದು. |
| **ಡೈನಾಮಿಕ್ ಅನ್ವೇಷಣೆ** | ಒಂದು ಒರ್ಕೆಸ್ಟ್ರೇಟರ್ ರನ್‌ಟೈಮ್‌ನಲ್ಲಿ ಏಜೆಂಟ್ ಕಾರ್ಡ್ ರೆಜಿಸ್ಟ್ರಿಯನ್ನು ಪ್ರಶ್ನಿಸಿ ನಿರ್ದಿಷ್ಟ ಉಪ-ಕಾರ್ಯಕ್ಕೆ ಅತ್ಯುತ್ತಮ ಹೊಂದಾಣಿಕೆಯ ತಜ್ಞರನ್ನು ಕಂಡುಹಿಡಿಯಬಹುದು. |
| **ಸ್ಟ್ರೀಮಿಂಗ್ & ಪುಷ್ ಸೂಚನೆಗಳು** | A2A ರಿಯಲ್-ಟೈಮ್ ಪ್ರಗತಿ ಅಪ್ಡೇಟ್‌ಗಳಿಗಾಗಿ Server-Sent Events (SSE) ಅನ್ನು ಬೆಂಬಲಿಸುತ್ತದೆ ಮತ್ತು ದೀರ್ಘಾವಧಿ ಕಾರ್ಯಗಳಿಗಾಗಿ ಪುಷ್ ಸೂಚನೆಗಳನ್ನು ಒದಗಿಸುತ್ತದೆ. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಕಲಿತವುಗಳು:

1. **A2A ಪ್ರೋಟೋಕಾಲ್ ಎಂದರೇನು** — ಏಜೆಂಟ್-ರಿಂದ ಏಜೆಂಟ್‌ವರೆಗೆ ಅನ್ವೇಷಣೆ, ಸಂದೇಶ ವಿನಿಮಯ ಮತ್ತು ಕಾರ್ಯ ನಿರ್ವಹಣೆಗೆ ಒಂದು ತೆರೆದ ಮಾನದಂಡ.
2. **ವಿಶೇಷಗೊಳಿಸಿದ ಏಜೆಂಟ್‌ಗಳನ್ನು ಹೇಗೆ ರಚಿಸುವುದು** — ಒಂದು ಕರೆನ್ಸಿ ವಿನিমಯ ಏಜೆಂಟ್, ಒಂದು ಚಟುವಟಿಕೆ ಯೋಜಕ ಏಜೆಂಟ್, ಮತ್ತು ಒಂದು ಪ್ರಯಾಣ ನಿರ್ವಹಣೆಯ ಆಯೋಜಕ.
3. **ಏಜೆಂಟ್‌ಗಳನ್ನು ವರ್ಕ್‌ಫ್ಲೋಗೆ ಜೋಡಿಸುವುದು ಹೇಗೆ** — ಎಜೆಂಟ್‌ಗಳ ನಡುವೆ ಕ್ರಮಬದ್ಧ ಸಂದೇಶ ವಹಿವಾಟನ್ನು ಮಾದರಿಯಾಗಿಸಲು `WorkflowBuilder` ಅನ್ನು ಬಳಸುವುದು.
4. **ಉತ್ಪಾದನೆಯಲ್ಲಿ A2A ಹೇಗೆ ಕೆಲಸ ಮಾಡುತ್ತದೆ** — ಡೈನಾಮಿಕ್ ಅನ್ವೇಷಣೆ ಮತ್ತು ಸ್ಟ್ರೀಮಿಂಗ್ ಅಪ್ಡೇಟ್‌ಗಳೊಂದಿಗೆ ಕ್ರಾಸ್-ಫ್ರೇಮ್‌ವರ್ಕ್ ಮತ್ತು ಕ್ರಾಸ್-ಸರ್ವಿಸ್ ಸಹಕಾರವನ್ನು ಸಕ್ರಿಯಗೊಳಿಸುವುದು.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ಜವಾಬ್ದಾರಿ ನಿರಾಕರಣೆ:

ಈ ದಸ್ತಾವೇಜನ್ನು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಕಾಯ್ದುಕೊಳ್ಳಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸೂಕ್ತತೆಗಳಿರಬಹುದು ಎಂಬುದನ್ನು ದಯವಿಟ್ಟು ಗಮನದಲ್ಲಿಡಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜನ್ನು ಅಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಗಂಭೀರ ಅಥವಾ ನಿರ್ಣಾಯಕ ಮಾಹಿತಿಗಾಗಿ ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ತಿಳುವಳಿಕೆಗಳು ಅಥವಾ ತಪ್ಪಾಗಿ ವ್ಯಾಖ್ಯಾನಿಸುವುದಕ್ಕಾಗಿ ನಾವು ಜವಾಬ್ದಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
